In [2]:
"""
Coordination Features - Data Diagnostic
========================================
快速检查数据是否适合提取coordination特征

运行时间: <1分钟
"""

import pandas as pd
import numpy as np
from collections import Counter
from datetime import timedelta

def main():
    print("\n" + "="*70)
    print("COORDINATION FEATURES - DATA DIAGNOSTIC")
    print("="*70)
    
    # ========== 加载数据 ==========
    print("\nLoading data...")
    
    try:
        df = pd.read_csv('reddit_wsb_for_network.csv')
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        print(f"✓ Loaded {len(df):,} items")
    except FileNotFoundError:
        print("✗ reddit_wsb_for_network.csv not found!")
        return
    
    # ========== 检查1: Text Duplication潜力 ==========
    print("\n" + "="*70)
    print("CHECK 1: TEXT DUPLICATION POTENTIAL")
    print("="*70)
    
    # 基础统计
    total_texts = len(df)
    unique_texts = df['text'].nunique()
    unique_ratio = unique_texts / total_texts * 100
    
    print(f"\nBasic statistics:")
    print(f"  Total texts: {total_texts:,}")
    print(f"  Unique texts: {unique_texts:,}")
    print(f"  Unique ratio: {unique_ratio:.1f}%")
    
    # 重复文本分析
    text_counts = df['text'].value_counts()
    duplicates = text_counts[text_counts > 1]
    
    print(f"\nDuplication analysis:")
    print(f"  Duplicated texts: {len(duplicates):,}")
    print(f"  Duplication rate: {(1 - unique_ratio/100)*100:.1f}%")
    
    if len(duplicates) > 0:
        print(f"\nTop duplicates:")
        for i, (text, count) in enumerate(duplicates.head(5).items(), 1):
            preview = text[:60] + "..." if len(text) > 60 else text
            print(f"  {i}. Count={count}: \"{preview}\"")
    
    # 评估
    print(f"\n{'='*70}")
    if unique_ratio > 95:
        print("⚠️  ASSESSMENT: Low duplication - Text duplication features may have weak signal")
    elif unique_ratio > 90:
        print("✓ ASSESSMENT: Moderate duplication - Text duplication features worth trying")
    else:
        print("✓✓ ASSESSMENT: High duplication - Text duplication features highly recommended!")
    print(f"{'='*70}")
    
    # ========== 检查2: Burstiness潜力 ==========
    print("\n" + "="*70)
    print("CHECK 2: BURSTINESS (TEMPORAL SYNCHRONY) POTENTIAL")
    print("="*70)
    
    # 创建时间窗口
    df['time_window_5min'] = df['timestamp'].dt.floor('5min')
    df['time_window_1min'] = df['timestamp'].dt.floor('1min')
    
    # 5分钟窗口统计
    volume_5min = df.groupby('time_window_5min').size()
    
    print(f"\n5-minute window statistics:")
    print(f"  Mean posts/5min: {volume_5min.mean():.1f}")
    print(f"  Std posts/5min: {volume_5min.std():.1f}")
    print(f"  Max posts/5min: {volume_5min.max()}")
    print(f"  Min posts/5min: {volume_5min.min()}")
    
    # 1分钟窗口统计
    volume_1min = df.groupby('time_window_1min').size()
    
    print(f"\n1-minute window statistics:")
    print(f"  Mean posts/1min: {volume_1min.mean():.1f}")
    print(f"  Std posts/1min: {volume_1min.std():.1f}")
    print(f"  Max posts/1min: {volume_1min.max()}")
    
    # 计算简单的burstiness indicator
    sync_ratio = volume_1min.max() / (volume_5min.mean() / 5)
    
    print(f"\nBurstiness indicators:")
    print(f"  Peak 1min / Avg per-min: {sync_ratio:.1f}x")
    
    # Inter-arrival time分析
    df_sorted = df.sort_values('timestamp')
    inter_arrivals = df_sorted['timestamp'].diff().dt.total_seconds()
    inter_arrivals = inter_arrivals[inter_arrivals.notna()]
    
    print(f"\nInter-arrival time (seconds):")
    print(f"  Mean: {inter_arrivals.mean():.1f}")
    print(f"  Std: {inter_arrivals.std():.1f}")
    print(f"  Min: {inter_arrivals.min():.1f}")
    print(f"  Median: {inter_arrivals.median():.1f}")
    print(f"  <1 second: {(inter_arrivals < 1).sum():,} ({(inter_arrivals < 1).sum()/len(inter_arrivals)*100:.1f}%)")
    print(f"  <5 seconds: {(inter_arrivals < 5).sum():,} ({(inter_arrivals < 5).sum()/len(inter_arrivals)*100:.1f}%)")
    
    # 评估
    print(f"\n{'='*70}")
    if sync_ratio > 5:
        print("✓✓ ASSESSMENT: Strong burst patterns - Burstiness features highly recommended!")
    elif sync_ratio > 3:
        print("✓ ASSESSMENT: Moderate burst patterns - Burstiness features recommended")
    else:
        print("⚠️  ASSESSMENT: Weak burst patterns - Burstiness features may have limited signal")
    print(f"{'='*70}")
    
    # ========== 检查3: User Overlap潜力 ==========
    print("\n" + "="*70)
    print("CHECK 3: USER OVERLAP POTENTIAL")
    print("="*70)
    
    # 按type分组
    if 'type' in df.columns:
        type_counts = df['type'].value_counts()
        print(f"\nContent type distribution:")
        for t, count in type_counts.items():
            print(f"  {t}: {count:,}")
        
        # 独立帖子数
        posts = df[df['type'] == 'post']
        unique_posts = posts['post_id'].nunique() if 'post_id' in posts.columns else len(posts)
        
        print(f"\nThread statistics:")
        print(f"  Unique posts (threads): {unique_posts:,}")
    else:
        print("\n⚠️  No 'type' column found")
        unique_posts = 0
    
    # 用户统计
    unique_users = df['user_id'].nunique()
    
    print(f"\nUser statistics:")
    print(f"  Unique users: {unique_users:,}")
    print(f"  Avg posts per user: {len(df)/unique_users:.1f}")
    
    # 多帖参与分析
    if unique_posts > 0:
        user_post_counts = df.groupby('user_id')['post_id'].nunique()
        multi_thread_users = (user_post_counts > 1).sum()
        
        print(f"\nMulti-thread participation:")
        print(f"  Users in >1 thread: {multi_thread_users:,} ({multi_thread_users/unique_users*100:.1f}%)")
        print(f"  Avg threads per user: {user_post_counts.mean():.1f}")
        print(f"  Max threads per user: {user_post_counts.max()}")
    
    # 评估
    print(f"\n{'='*70}")
    if unique_posts > 20 and multi_thread_users > 100:
        print("✓✓ ASSESSMENT: Sufficient threads & user overlap - Overlap features recommended!")
    elif unique_posts > 10:
        print("✓ ASSESSMENT: Moderate thread count - Overlap features worth trying")
    else:
        print("⚠️  ASSESSMENT: Few unique threads - Overlap features may have weak signal")
    print(f"{'='*70}")
    
    # ========== 检查4: Text Length分布 ==========
    print("\n" + "="*70)
    print("CHECK 4: TEXT LENGTH DISTRIBUTION")
    print("="*70)
    
    text_lengths = df['text'].str.len()
    
    print(f"\nText length statistics:")
    print(f"  Mean: {text_lengths.mean():.1f} chars")
    print(f"  Median: {text_lengths.median():.0f} chars")
    print(f"  Max: {text_lengths.max()} chars")
    print(f"  At limit (500): {(text_lengths >= 500).sum():,} ({(text_lengths >= 500).sum()/len(df)*100:.1f}%)")
    
    # 长度分布
    print(f"\nLength distribution:")
    print(f"  <100 chars: {(text_lengths < 100).sum():,} ({(text_lengths < 100).sum()/len(df)*100:.1f}%)")
    print(f"  100-200: {((text_lengths >= 100) & (text_lengths < 200)).sum():,} ({((text_lengths >= 100) & (text_lengths < 200)).sum()/len(df)*100:.1f}%)")
    print(f"  200-500: {((text_lengths >= 200) & (text_lengths < 500)).sum():,} ({((text_lengths >= 200) & (text_lengths < 500)).sum()/len(df)*100:.1f}%)")
    print(f"  =500 (truncated): {(text_lengths >= 500).sum():,} ({(text_lengths >= 500).sum()/len(df)*100:.1f}%)")
    
    # 评估
    truncation_rate = (text_lengths >= 500).sum() / len(df) * 100
    
    print(f"\n{'='*70}")
    if truncation_rate < 10:
        print("✓✓ ASSESSMENT: Low truncation rate - Text features should work well")
    elif truncation_rate < 30:
        print("✓ ASSESSMENT: Moderate truncation - Text features acceptable with caveat")
    else:
        print("⚠️  ASSESSMENT: High truncation - Text features may be impacted")
    print(f"{'='*70}")
    
    # ========== 检查5: 时间分布 ==========
    print("\n" + "="*70)
    print("CHECK 5: TEMPORAL DISTRIBUTION")
    print("="*70)
    
    print(f"\nDate range:")
    print(f"  Start: {df['timestamp'].min()}")
    print(f"  End: {df['timestamp'].max()}")
    print(f"  Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
    
    # 按日统计
    daily_volume = df.groupby(df['timestamp'].dt.date).size()
    
    print(f"\nDaily volume:")
    print(f"  Mean: {daily_volume.mean():.0f} posts/day")
    print(f"  Peak: {daily_volume.max()} posts/day")
    print(f"  Peak date: {daily_volume.idxmax()}")
    
    # ========== 最终建议 ==========
    print("\n" + "="*70)
    print("FINAL RECOMMENDATION")
    print("="*70)
    
    scores = {
        'text_duplication': 100 - unique_ratio,
        'burstiness': min(sync_ratio * 10, 50),
        'user_overlap': min(multi_thread_users / unique_users * 100, 50) if unique_posts > 0 else 0
    }
    
    overall_score = sum(scores.values())
    
    print(f"\nFeature potential scores (0-100):")
    print(f"  Text Duplication: {scores['text_duplication']:.1f}")
    print(f"  Burstiness: {scores['burstiness']:.1f}")
    print(f"  User Overlap: {scores['user_overlap']:.1f}")
    print(f"  Overall: {overall_score:.1f}")
    
    print(f"\n{'='*70}")
    if overall_score > 80:
        print("✓✓ STRONG RECOMMENDATION: Implement ALL coordination features!")
        print("   Expected value: Very High")
    elif overall_score > 50:
        print("✓ RECOMMENDATION: Implement Burstiness + Text Duplication")
        print("   Expected value: High")
    else:
        print("⚠️  MODERATE RECOMMENDATION: Try Burstiness first")
        print("   Expected value: Medium")
    print(f"{'='*70}")
    
    # 具体建议
    print(f"\nNext steps:")
    if scores['text_duplication'] > 5:
        print("  ✓ Implement Text Duplication features")
    if scores['burstiness'] > 20:
        print("  ✓ Implement Burstiness features")
    if scores['user_overlap'] > 10:
        print("  ✓ Implement User Overlap features")
    
    print(f"\nEstimated implementation time:")
    total_time = 0
    if scores['text_duplication'] > 5:
        print("  - Text Duplication: 2-3 hours")
        total_time += 2.5
    if scores['burstiness'] > 20:
        print("  - Burstiness: 1-2 hours")
        total_time += 1.5
    if scores['user_overlap'] > 10:
        print("  - User Overlap: 3-4 hours")
        total_time += 3.5
    
    if total_time > 0:
        print(f"  Total: ~{total_time:.0f}-{total_time+2:.0f} hours")
    
    print("\n" + "="*70)


if __name__ == "__main__":
    main()


COORDINATION FEATURES - DATA DIAGNOSTIC

Loading data...
✓ Loaded 1,606,093 items

CHECK 1: TEXT DUPLICATION POTENTIAL

Basic statistics:
  Total texts: 1,606,093
  Unique texts: 1,514,046
  Unique ratio: 94.3%

Duplication analysis:
  Duplicated texts: 26,975
  Duplication rate: 5.7%

Top duplicates:
  1. Count=5824: "GME"
  2. Count=2019: "Gme"
  3. Count=1504: "GME [removed]"
  4. Count=1030: "Thanks for participating on WSB! 

We're experiencing very h..."
  5. Count=530: "GME 🚀🚀🚀"

✓ ASSESSMENT: Moderate duplication - Text duplication features worth trying

CHECK 2: BURSTINESS (TEMPORAL SYNCHRONY) POTENTIAL

5-minute window statistics:
  Mean posts/5min: 20.8
  Std posts/5min: 62.0
  Max posts/5min: 1261
  Min posts/5min: 1

1-minute window statistics:
  Mean posts/1min: 6.8
  Std posts/1min: 15.5
  Max posts/1min: 273

Burstiness indicators:
  Peak 1min / Avg per-min: 65.7x

Inter-arrival time (seconds):
  Mean: 39.3
  Std: 692.9
  Min: 0.0
  Median: 2.0
  <1 second: 357,299 (22